In [1]:
import torch

inputs = torch.tensor(
    [[0.72, 0.45, 0.31], # Dream  (x^1)
     [0.75, 0.20, 0.55], # big    (x^2)
     [0.30, 0.80, 0.40], # and    (x^3)
     [0.85, 0.35, 0.60], # work   (x^4)
     [0.55, 0.15, 0.75], # for    (x^5)
     [0.25, 0.20, 0.85]] # it     (x^6)
)

# correspinding words
words = ['Dream', 'big', 'and', 'work', 'for', 'it']

In [2]:
import torch.nn as nn

class CausalAttention(nn.Module):

    def __init__(self, d_in, d_out, context_length, dropout, qkv_bias=False):
        super().__init__()
        self.d_out = d_out
        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias) 
        self.dropout = nn.Dropout(dropout)
        self.register_buffer("mask", torch.triu(torch.ones(context_length, context_length), diagonal=1).bool())

    def forward(self, x):
        b, num_tokens, d_in = x.size()

        queries = self.W_query(x) 
        keys = self.W_key(x)
        values = self.W_value(x)

        attn_scores = queries @ keys.transpose(1, 2)
        attn_scores.masked_fill_(self.mask.bool()[:num_tokens, :num_tokens], -torch.inf)
        attn_weights = torch.softmax(attn_scores / keys.shape[-1] ** 0.5, dim=-1)

        attn_weights = self.dropout(attn_weights)

        context_vec = attn_weights @ values

        return context_vec

In [3]:
d_in = inputs.shape[-1]
d_out = 2

In [4]:
batch = torch.stack((inputs, inputs), dim=0) # (batch, num_tokens, d_in)
print(batch.shape)

torch.Size([2, 6, 3])


In [5]:
class MultiHeadAttentionWrapper(nn.Module):

    def __init__(self, d_in, d_out, context_length, dropout, num_heads, qkv_bias=False):
        super().__init__()
        self.heads = nn.ModuleList(
            [CausalAttention(d_in, d_out, context_length, dropout, qkv_bias)
                for _ in range(num_heads)]
        )

    def forward(self, x):
        return torch.cat([head(x) for head in self.heads], dim=-1)

In [6]:
torch.manual_seed(123)
context_length = batch.shape[1]
d_in, d_out = 3, 2
mha = MultiHeadAttentionWrapper(d_in, d_out, context_length, dropout=0.0, num_heads=2)

In [7]:
context_vecs = mha(batch)
print(context_vecs)
print("\nContext vectors shape:", context_vecs.shape) # (batch, num_tokens, d_out * num_heads)

tensor([[[-0.5762, -0.1627,  0.5569,  0.3635],
         [-0.5650, -0.0630,  0.5599,  0.3006],
         [-0.5472, -0.1226,  0.5285,  0.3435],
         [-0.5787, -0.0943,  0.5621,  0.3388],
         [-0.5593, -0.0436,  0.5509,  0.3046],
         [-0.5287, -0.0033,  0.5277,  0.2743]],

        [[-0.5762, -0.1627,  0.5569,  0.3635],
         [-0.5650, -0.0630,  0.5599,  0.3006],
         [-0.5472, -0.1226,  0.5285,  0.3435],
         [-0.5787, -0.0943,  0.5621,  0.3388],
         [-0.5593, -0.0436,  0.5509,  0.3046],
         [-0.5287, -0.0033,  0.5277,  0.2743]]], grad_fn=<CatBackward0>)

Context vectors shape: torch.Size([2, 6, 4])


MULTI-HEAD ATTENTION WITH WEIGHT SPLITS

Step-1: Inputs

In [8]:
import torch

# Input: (batch=1, seq_len=3, d_model=6)
x = torch.tensor([[
    [1.0, 2.0, 3.0, 4.0, 5.0, 6.0],
    [6.0, 5.0, 4.0, 3.0, 2.0, 1.0],
    [1.0, 1.0, 1.0, 1.0, 1.0, 1.0]
]])

batch_size, seq_len, d_model = x.shape

Step-2: Define d_out and num_heads
Step-3: Initialize W_q, W_k, W_v
Step-4: Calculate Q, K, V

In [9]:
# Define 6x6 projection matrices for queries, keys, and values (d_model x d_model)
torch.manual_seed(0)
W_query = torch.randn(d_model, d_model)
W_key = torch.randn(d_model, d_model)
W_value = torch.randn(d_model, d_model)

# Compute Q, K, V
# Shape: (batch, seq_len, d_model) @ (d_model, d_model) -> (batch, seq_len, d_model)
Q = x @ W_query
K = x @ W_key
V = x @ W_value

# Print Q, K, V
print("Q:\n", Q)
print("K:\n", K)
print("V:\n", V)

# Dimensions
print("x shape:", x.shape)             # (1, 3, 6)
print("Wq shape:", W_query.shape)      # (6, 6)
print("Wk shape:", W_key.shape)        # (6, 6)
print("Wv shape:", W_value.shape)      # (6, 6)
print("Q shape:", Q.shape)             # (1, 3, 6)
print("K shape:", K.shape)             # (1, 3, 6)
print("V shape:", V.shape)             # (1, 3, 6)

Q:
 tensor([[[ -9.0244, -11.7287,  15.5360,  -1.4474,  -4.5326,   9.4674],
         [ -8.0564, -13.2309,   8.2228,  -8.9680,   3.1995,   4.8321],
         [ -2.4401,  -3.5657,   3.3941,  -1.4879,  -0.1904,   2.0428]]])
K:
 tensor([[[  8.2602,  14.1116,  -5.0345, -16.4865,  -2.9948,   8.3139],
         [ -6.1188,  -0.1587,  -5.0885, -14.3014,   4.9540,   5.6093],
         [  0.3059,   1.9933,  -1.4461,  -4.3983,   0.2799,   1.9890]]])
V:
 tensor([[[ 0.5076, -3.4353,  1.8576,  2.8041,  8.9427, 13.1841],
         [-1.9113, -3.6934,  1.8502,  1.7622,  1.6981,  3.0978],
         [-0.2005, -1.0184,  0.5297,  0.6523,  1.5201,  2.3260]]])
x shape: torch.Size([1, 3, 6])
Wq shape: torch.Size([6, 6])
Wk shape: torch.Size([6, 6])
Wv shape: torch.Size([6, 6])
Q shape: torch.Size([1, 3, 6])
K shape: torch.Size([1, 3, 6])
V shape: torch.Size([1, 3, 6])


Step-5: Unrolling the last dimension i.e splitting into 2 heads

In [10]:
num_heads = 2
head_dim = 3

Q = Q.view(1, 3, num_heads, head_dim) # Reshape Q to (batch, seq_len, num_heads, head_dim) 
K = K.view(1, 3, num_heads, head_dim)
V = V.view(1, 3, num_heads, head_dim)

print("Q after unrolling:\n", Q)
print("K after unrolling:\n", K)
print("V after unrolling:\n", V)

Q after unrolling:
 tensor([[[[ -9.0244, -11.7287,  15.5360],
          [ -1.4474,  -4.5326,   9.4674]],

         [[ -8.0564, -13.2309,   8.2228],
          [ -8.9680,   3.1995,   4.8321]],

         [[ -2.4401,  -3.5657,   3.3941],
          [ -1.4879,  -0.1904,   2.0428]]]])
K after unrolling:
 tensor([[[[  8.2602,  14.1116,  -5.0345],
          [-16.4865,  -2.9948,   8.3139]],

         [[ -6.1188,  -0.1587,  -5.0885],
          [-14.3014,   4.9540,   5.6093]],

         [[  0.3059,   1.9933,  -1.4461],
          [ -4.3983,   0.2799,   1.9890]]]])
V after unrolling:
 tensor([[[[ 0.5076, -3.4353,  1.8576],
          [ 2.8041,  8.9427, 13.1841]],

         [[-1.9113, -3.6934,  1.8502],
          [ 1.7622,  1.6981,  3.0978]],

         [[-0.2005, -1.0184,  0.5297],
          [ 0.6523,  1.5201,  2.3260]]]])


Step-6: Grouping by heads

In [11]:
Q = Q.transpose(1, 2)
K = K.transpose(1, 2)
V = V.transpose(1, 2)

print("Q after grouping by heads:\n", Q)
print("K after grouping by heads:\n", K)
print("V after grouping by heads:\n", V)

Q after grouping by heads:
 tensor([[[[ -9.0244, -11.7287,  15.5360],
          [ -8.0564, -13.2309,   8.2228],
          [ -2.4401,  -3.5657,   3.3941]],

         [[ -1.4474,  -4.5326,   9.4674],
          [ -8.9680,   3.1995,   4.8321],
          [ -1.4879,  -0.1904,   2.0428]]]])
K after grouping by heads:
 tensor([[[[  8.2602,  14.1116,  -5.0345],
          [ -6.1188,  -0.1587,  -5.0885],
          [  0.3059,   1.9933,  -1.4461]],

         [[-16.4865,  -2.9948,   8.3139],
          [-14.3014,   4.9540,   5.6093],
          [ -4.3983,   0.2799,   1.9890]]]])
V after grouping by heads:
 tensor([[[[ 0.5076, -3.4353,  1.8576],
          [-1.9113, -3.6934,  1.8502],
          [-0.2005, -1.0184,  0.5297]],

         [[ 2.8041,  8.9427, 13.1841],
          [ 1.7622,  1.6981,  3.0978],
          [ 0.6523,  1.5201,  2.3260]]]])


Step-7: Attention Score (Transposing 'K' i.e. last 2 dims)

In [12]:
K_T = K.transpose(2, 3) # Transpose K to (batch, num_heads, head_dim, seq_len)
print("K_T shape:", K_T)

K_T shape: tensor([[[[  8.2602,  -6.1188,   0.3059],
          [ 14.1116,  -0.1587,   1.9933],
          [ -5.0345,  -5.0885,  -1.4461]],

         [[-16.4865, -14.3014,  -4.3983],
          [ -2.9948,   4.9540,   0.2799],
          [  8.3139,   5.6093,   1.9890]]]])


Step-8: Find Attention Scores

In [13]:
attn_scores = Q @ K_T # (batch, num_heads, seq_len, head_dim) @ (batch, num_heads, head_dim, seq_len) -> (batch, num_heads, seq_len, seq_len)
print("Attention scores shape:", attn_scores.shape) # (batch, num_heads, seq_len, seq_len)
print("Attention scores:\n", attn_scores)

Attention scores shape: torch.Size([1, 2, 3, 3])
Attention scores:
 tensor([[[[-318.2692,  -21.9748,  -48.6063],
          [-294.6535,    9.5538,  -40.7285],
          [ -87.5604,   -1.7744,  -12.7621]],

         [[ 116.1476,   51.3506,   23.9283],
          [ 178.4425,  171.2106,   49.9505],
          [  42.0843,   31.7945,   10.5541]]]])


Step-9: Apply Causal Mask

In [14]:
mask = torch.triu(torch.ones(seq_len, seq_len), diagonal=1).bool()
print("Causal mask:\n", mask)

attn_scores.masked_fill_(mask, -torch.inf)
print("Attention scores after applying causal mask:\n", attn_scores)

Causal mask:
 tensor([[False,  True,  True],
        [False, False,  True],
        [False, False, False]])
Attention scores after applying causal mask:
 tensor([[[[-318.2692,      -inf,      -inf],
          [-294.6535,    9.5538,      -inf],
          [ -87.5604,   -1.7744,  -12.7621]],

         [[ 116.1476,      -inf,      -inf],
          [ 178.4425,  171.2106,      -inf],
          [  42.0843,   31.7945,   10.5541]]]])


In [15]:
torch.set_printoptions(precision=3, sci_mode=False)
head_dim = 3
attn_weights = torch.softmax(attn_scores / head_dim ** 0.5, dim=-1)
print("Attention weights shape:", attn_weights.shape) # (batch, num_heads, seq_len, seq_len)
print("Attention weights:\n", attn_weights)

Attention weights shape: torch.Size([1, 2, 3, 3])
Attention weights:
 tensor([[[[    1.000,     0.000,     0.000],
          [    0.000,     1.000,     0.000],
          [    0.000,     0.998,     0.002]],

         [[    1.000,     0.000,     0.000],
          [    0.985,     0.015,     0.000],
          [    0.997,     0.003,     0.000]]]])


In [16]:
dropout = nn.Dropout(0.1)
attn_weights = dropout(attn_weights)
print("Attention weights after dropout:\n", attn_weights)

Attention weights after dropout:
 tensor([[[[    1.111,     0.000,     0.000],
          [    0.000,     1.111,     0.000],
          [    0.000,     1.109,     0.002]],

         [[    1.111,     0.000,     0.000],
          [    1.094,     0.017,     0.000],
          [    1.108,     0.003,     0.000]]]])


Step-10: Calculate Context Vectors

In [17]:
context_vec = attn_weights @ V # (batch, num_heads, seq_len, seq_len) @ (batch, num_heads, seq_len, head_dim) -> (batch, num_heads, seq_len, head_dim)
print("Context vectors shape:", context_vec.shape) # (batch, num_heads, seq_len, head_dim)
print("Context vectors:\n", context_vec)

Context vectors shape: torch.Size([1, 2, 3, 3])
Context vectors:
 tensor([[[[ 0.564, -3.817,  2.064],
          [-2.124, -4.104,  2.056],
          [-2.120, -4.099,  2.053]],

         [[ 3.116,  9.936, 14.649],
          [ 3.098,  9.814, 14.479],
          [ 3.113,  9.915, 14.620]]]])


Step-11: Reformat and Concatenate

In [18]:
context_vec = context_vec.transpose(1, 2)
print("Context vectors after transposing 1 and 2:\n", context_vec)

Context vectors after transposing 1 and 2:
 tensor([[[[ 0.564, -3.817,  2.064],
          [ 3.116,  9.936, 14.649]],

         [[-2.124, -4.104,  2.056],
          [ 3.098,  9.814, 14.479]],

         [[-2.120, -4.099,  2.053],
          [ 3.113,  9.915, 14.620]]]])


In [19]:
# Concatenate heads
context_vec = context_vec.reshape(batch_size, seq_len, num_heads * head_dim)
print("Context vectors shape after concatenating heads:", context_vec.shape) # (batch, seq_len, num_heads * head_dim)
print("Context vectors after concatenating heads:\n", context_vec)

Context vectors shape after concatenating heads: torch.Size([1, 3, 6])
Context vectors after concatenating heads:
 tensor([[[ 0.564, -3.817,  2.064,  3.116,  9.936, 14.649],
         [-2.124, -4.104,  2.056,  3.098,  9.814, 14.479],
         [-2.120, -4.099,  2.053,  3.113,  9.915, 14.620]]])


Implementing MHA in a class (11 steps)

In [20]:
class MultiHeadAttention(nn.Module):

    def __init__(self, d_in, d_out, context_length, dropout, num_heads, qkv_bias=False):
        super().__init__()
        assert (d_out % num_heads == 0), "d_out must be divisible by num_heads"

        self.d_out = d_out
        self.num_heads = num_heads
        self.head_dim = d_out // num_heads

        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)

        self.dropout = nn.Dropout(dropout)
        self.out_proj = nn.Linear(d_out, d_out)

        self.register_buffer(
            "mask", 
            torch.triu(torch.ones(context_length, context_length), diagonal=1).bool()
        )

    def forward(self, x):
        b, num_tokens, d_in = x.size()

        Q = self.W_query(x)
        K = self.W_key(x)
        V = self.W_value(x)

        # We implicitly split the matrix by adding a 'num_heads' dimension
        # Unroll the last dim: (b, num_tokens, d_out) -> (b, num_tokens, num_heads, head_dim)
        Q = Q.view(b, num_tokens, self.num_heads, self.head_dim)
        K = K.view(b, num_tokens, self.num_heads, self.head_dim)
        V = V.view(b, num_tokens, self.num_heads, self.head_dim)

        # Transpose to group by heads: (b, num_tokens, num_heads, head_dim) -> (b, num_heads, num_tokens, head_dim)
        Q = Q.transpose(1, 2)
        K = K.transpose(1, 2)
        V = V.transpose(1, 2)

        # Compute self-attention scores
        attn_scores = Q @ K.transpose(-2, -1) # (b, num_heads, num_tokens, head_dim) @ (b, num_heads, head_dim, num_tokens) -> (b, num_heads, num_tokens, num_tokens)

        # Original mask truncated to the current number of tokens
        mask_bool = self.mask.bool()[:num_tokens, :num_tokens]

        # Apply causal mask to attention scores
        attn_scores.masked_fill_(mask_bool, -torch.inf)

        # Compute attention weights
        attn_weights = torch.softmax(attn_scores / K.shape[-1] ** 0.5, dim=-1)
        attn_weights = self.dropout(attn_weights)

        # Compute context vectors 
        # Shape: (b, num_heads, num_tokens, num_tokens) @ (b, num_heads, num_tokens, head_dim) -> (b, num_heads, num_tokens, head_dim)
        context_vec = attn_weights @ V

        # Transpose back to (b, num_tokens, num_heads, head_dim)
        context_vec = context_vec.transpose(1, 2)

        # Concatenate heads: (b, num_tokens, num_heads, head_dim) -> (b, num_tokens, d_out)
        # context_vec = context_vec.reshape(b, num_tokens, self.head_dim * self.num_heads)
        context_vec = context_vec.contiguous().view(b, num_tokens, self.d_out)
        context_vec = self.out_proj(context_vec) # Optional projection

        return context_vec

In [21]:
torch.manual_seed(123)

# Define input tensor
inputs = torch.tensor(
    [[0.43, 0.15, 0.89, 0.55, 0.87, 0.12],
     [0.75, 0.20, 0.55, 0.30, 0.80, 0.40],
     [0.85, 0.35, 0.60, 0.25, 0.15, 0.75]]
)

batch = torch.stack([inputs, inputs], dim=0) # (batch, num_tokens, d_in)
print("Batch shape:", batch.shape) # (batch, num_tokens, d_in)
print("Batch:\n", batch)

batch_size, seq_len, d_in = batch.shape

d_out = 6
context_length = inputs.shape[0]
dropout = 0.0

mha = MultiHeadAttention(d_in, d_out, context_length, dropout, num_heads=2)

context_vecs = mha(batch)
print(context_vecs)
print("\nContext vectors shape:", context_vecs.shape) # (batch, num_tokens, d_out)

Batch shape: torch.Size([2, 3, 6])
Batch:
 tensor([[[0.430, 0.150, 0.890, 0.550, 0.870, 0.120],
         [0.750, 0.200, 0.550, 0.300, 0.800, 0.400],
         [0.850, 0.350, 0.600, 0.250, 0.150, 0.750]],

        [[0.430, 0.150, 0.890, 0.550, 0.870, 0.120],
         [0.750, 0.200, 0.550, 0.300, 0.800, 0.400],
         [0.850, 0.350, 0.600, 0.250, 0.150, 0.750]]])
tensor([[[ 0.232, -0.126,  0.081, -0.010, -0.286, -0.299],
         [ 0.202, -0.089,  0.069, -0.058, -0.252, -0.262],
         [ 0.133, -0.052,  0.019, -0.073, -0.257, -0.258]],

        [[ 0.232, -0.126,  0.081, -0.010, -0.286, -0.299],
         [ 0.202, -0.089,  0.069, -0.058, -0.252, -0.262],
         [ 0.133, -0.052,  0.019, -0.073, -0.257, -0.258]]],
       grad_fn=<ViewBackward0>)

Context vectors shape: torch.Size([2, 3, 6])
